### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="musk",
    dataset_year="1994",
    domain_str="chemistry & material science",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C51608",
    download_description="""
wget https://archive.ics.uci.edu/static/public/75/musk+version+2.zip && unzip musk+version+2.zip clean2.data.Z && uncompress clean2.data.Z && rm musk+version+2.zip && mkdir -p local-data-warehouse/musk && mv clean2.data local-data-warehouse/musk/
""",
    # References
    academic_reference_bibtex="""@article{dietterich1993comparison,
  title={A comparison of dynamic reposing and tangent distance for drug activity prediction},
  author={Dietterich, Thomas and Jain, Ajay and Lathrop, Richard and Lozano-Perez, Tomas},
  journal={Advances in neural information processing systems},
  volume={6},
  year={1993}
}
""",
    academic_reference_bibtex_key="dietterich1993comparison",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
- We rename the molecule IDs to remove the target leakage from the names.
- We drop the conformation name as it leaks information that the real task should not have (the correlation between specific conformations across samples).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="class",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="class",
    group_on="molecule_name"
)

## Preprocessing

In [2]:
import pandas as pd
import uuid

df = pd.read_csv(dataset_mold.path / "clean2.data", header=None, names=[
    "molecule_name", "conformation_name", *[f"feature_{i}" for i in range(166)], "class",
])
print("Loaded data shape:", df.shape)

df = df.drop(columns=["conformation_name"])

df["class"] = df["class"].map({0: "non-musk", 1: "musk"})

# Create mapping: molecule -> random string id
mapping = {val: uuid.uuid4().hex[:12] for val in df["molecule_name"].unique()}
df["molecule_name"] = df["molecule_name"].map(mapping)

as_cat_type = ["molecule_name", "class"]
df[as_cat_type] = df[as_cat_type].astype("category")


df = df.sample(frac=1, random_state=42).sort_values(by="molecule_name").reset_index(drop=True)

Loaded data shape: (6598, 169)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 6,598
Columns: 168
Use sampling: False (sample size: 6,598)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['feature_126', 'feature_72', 'feature_33', 'feature_20', 'feature_132', 'feature_55', 'feature_128', 'feature_139', 'feature_59', 'feature_47']
Rows remaining as candidates after top-10 filter: 860 (of 6,598)

#### Duplicate Report
Total duplicate rows: 17 (0.26% of dataset)
Duplicate rows ignoring target: 17 (0.26% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,molecule_name,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_49,feature_50,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,feature_79,feature_80,feature_81,feature_82,feature_83,feature_84,feature_85,feature_86,feature_87,feature_88,feature_89,feature_90,feature_91,feature_92,feature_93,feature_94,feature_95,feature_96,feature_97,feature_98,feature_99,feature_100,feature_101,feature_102,feature_103,feature_104,feature_105,feature_106,feature_107,feature_108,feature_109,feature_110,feature_111,feature_112,feature_113,feature_114,feature_115,feature_116,feature_117,feature_118,feature_119,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128,feature_129,feature_130,feature_131,feature_132,feature_133,feature_134,feature_135,feature_136,feature_137,feature_138,feature_139,feature_140,feature_141,feature_142,feature_143,feature_144,feature_145,feature_146,feature_147,feature_148,feature_149,feature_150,feature_151,feature_152,feature_153,feature_154,feature_155,feature_156,feature_157,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165,class
0,029079f90035,20,-190,-126,32,-117,-55,49,47,-32,-63,-47,-30,-20,-75,-94,-293,56,79,-8,-43,-20,-17,14,-63,-97,59,66,-122,-3,-132,-116,-5,63,-201,75,53,-175,10,-145,72,-162,-102,-28,-72,-87,-117,-95,-16,-4,-74,-50,-18,-1,64,-84,96,99,-139,3,-123,-60,-182,40,-186,-27,12,-165,-43,-133,-2,-142,-150,-27,-47,-36,-181,5,-2,-11,65,-13,10,-47,-35,-153,22,-55,-15,-17,-145,-202,9,44,-87,40,92,-163,28,-155,94,-188,21,-40,-55,-112,21,-152,-15,-61,-90,-52,3,23,18,-165,-55,-93,38,75,29,-114,-82,-96,44,-70,25,-83,-163,-68,-8,51,106,22,-4,21,-95,47,-85,31,56,-26,41,65,21,-178,-103,-122,-15,-38,79,123,-12,56,-169,-19,-185,-233,-240,-206,-132,-96,-15,137,-59,-120,27,musk
1,029079f90035,2,-195,-148,28,-117,-115,64,-2,62,-117,-47,90,-52,-68,-68,-299,54,-84,135,-30,80,80,137,-50,-97,-4,40,-93,-36,-146,-116,-19,75,-229,82,46,-175,18,-145,31,-175,-137,-71,-63,34,-115,-83,9,112,-98,43,93,116,81,-51,-62,91,-142,-50,-165,107,-188,1,-190,-13,25,-165,-67,-133,-63,-160,-149,-46,37,-59,-183,-11,-62,119,56,122,58,78,-55,-179,44,-98,-122,8,-149,-201,19,56,-123,45,-26,-33,29,-155,77,-197,-5,-53,-52,-65,-130,-169,-88,-67,41,20,133,156,151,-147,-12,-91,41,84,18,-116,-93,-113,101,15,4,-146,-188,-118,-51,58,-118,-26,13,-86,-59,65,-39,97,29,79,5,48,17,-178,-103,-118,-75,-90,45,123,-18,53,-169,-120,-191,-241,-259,-211,-129,4,8,142,-51,-109,77,musk
2,029079f90035,19,-187,-132,32,-117,-57,60,-10,39,-80,-38,105,-26,-20,-68,-283,51,-50,110,-41,68,82,144,-66,-97,3,65,-75,-3,-131,-116,-5,64,-187,73,53,-175,4,-145,58,-161,-101,-39,-76,53,-91,-100,-39,44,-92,62,85,130,49,-84,-38,94,-127,3,-131,-55,-182,40,-185,-27,12,-165,-62,-133,-17,-145,-150,-27,73,-53,-180,4,-86,115,64,105,62,78,-33,-162,43,-47,-137,-17,-143,-202,9,48,-76,40,92,-164,32,-157,83,-183,21,-51,-59,-46,-125,-159,-90,-69,41,0,131,150,156,-171,-59,-95,16,77,40,-114,-82,-96,45,-70,26,-88,-156,-66,-9,51,105,-10,64,-10,-98,48,-87,117,27,72,22,49,17,-178,-103,-123,-24,-39,71,123,-12,57,-168,-19,-184,-231,-200,-205,-132,-96,-15,137,-60,-120,26,musk
3,029079f90035,3,-190,-126,29,-117,-118,49,48,-32,-63,-48,-30,-20,-75,-94,-294,56,79,-7,-44,-20,-17,14

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,molecule_name,category,0.0,0.0,102.0,"98b23c5d54b3, 61371428a811, 9dfc2a4eee43, 8e7f05f58cc9, 788c7ba58c36, 2282258508e3, 976c2386baee, a5a7c7738afb, fc4fa93d903f, aa67a38891f9"
1,class,category,0.0,0.0,2.0,"non-musk, musk"
2,feature_0,int64,0.0,0.0,202.0,"44, 43, 35, 36, 46, 51, 37, 48, 47, 57"
3,feature_1,int64,0.0,0.0,260.0,"-198, -194, -199, -193, -195, -192, -196, -197, 86, -191"
4,feature_2,int64,0.0,0.0,221.0,"-145, -144, -112, -19, -22, -111, 31, -62, -23, -146"
5,feature_3,int64,0.0,0.0,257.0,"-76, -77, -69, 28, -70, 29, 33, 131, 32, 152"
6,feature_4,int64,0.0,0.0,129.0,"-117, -116, -115, -113, -112, -111, -114, -108, -110, -109"
7,feature_5,int64,0.0,0.0,358.0,"11, 10, 12, 86, 85, 54, -154, 55, 53, 52"
8,feature_6,int64,0.0,0.0,323.0,"56, 26, 57, -163, 27, -160, -162, -161, -164, -159"
9,feature_7,int64,0.0,0.0,389.0,"-95, -96, -103, 57, 64, -3, -171, -102, -5, 67"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
feature_0,6598.0,58.945135,53.249007,-31.0,292.0
feature_1,6598.0,-119.128524,90.813375,-199.0,95.0
feature_2,6598.0,-73.146560,67.956235,-167.0,81.0
feature_3,6598.0,-0.628372,80.444617,-114.0,161.0
feature_4,6598.0,-103.533495,64.387559,-118.0,325.0
feature_5,6598.0,18.359806,80.593655,-183.0,200.0
feature_6,6598.0,-14.108821,115.315673,-171.0,220.0
feature_7,6598.0,-1.858290,90.372537,-225.0,320.0
feature_8,6598.0,-86.003031,108.326676,-245.0,147.0
feature_9,6598.0,-44.495756,72.088903,-286.0,231.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column        rank                            
class         1         non-musk   5581  84.59
              2             musk   1017  15.41
molecule_name 1     98b23c5d54b3   1044  15.82
              2     61371428a811   1010  15.31
              3     9dfc2a4eee43    911  13.81
              4     8e7f05f58cc9    383   5.80
              5     788c7ba58c36    344   5.21

In [8]:
# Target Distribution
target_df

,count,pct
class,,
non-musk,5581,84.59
musk,1017,15.41


## Task Curation

In [9]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from sklearn.model_selection import StratifiedGroupKFold

N_REPEATS = 20
N_FOLDS = 3

splits = {}
for repeat_i in range(N_REPEATS):
    print_once = False

    splits[repeat_i] = {}

    sklearn_splits = StratifiedGroupKFold(n_splits=N_FOLDS, random_state=42 + repeat_i, shuffle=True).split(
        X=df,
        y=df[task_mold.target_column_name],
        groups=df[task_mold.group_on],
    )
    for fold_idx, (train_index, test_index) in enumerate(sklearn_splits):
        # Print len, target col count, and group counts
        train_data = df.iloc[train_index]
        test_data = df.iloc[test_index]

        if not print_once:
            print(f"""Train N: {len(train_index)}, Test N: {len(test_index)}
            Target Distribution:
            \tTrain target distribution: {df.iloc[train_index][task_mold.target_column_name].value_counts(normalize=True).to_dict()}
            \tTest target distribution: {df.iloc[test_index][task_mold.target_column_name].value_counts(normalize=True).to_dict()}
            Group Distribution {task_mold.group_on}:
            \tTrain: {len(train_data[task_mold.group_on].unique())}
            \tTest: {len(test_data[task_mold.group_on].unique())}
            """
            )
            print_once = True
        splits[repeat_i][fold_idx] = (train_index.tolist(), test_index.tolist())

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create stratified grouped 20-repeated 3-fold split. This creates ca. 30 group members (500-3000 samples) per test set.",
    splits=splits
)

Train N: 5638, Test N: 960
            Target Distribution:
            	Train target distribution: {'non-musk': 0.8852429939694927, 'musk': 0.11475700603050727}
            	Test target distribution: {'non-musk': 0.6145833333333334, 'musk': 0.3854166666666667}
            Group Distribution molecule_name:
            	Train: 70
            	Test: 32
            
Train N: 3708, Test N: 2890
            Target Distribution:
            	Train target distribution: {'non-musk': 0.7820927723840345, 'musk': 0.21790722761596548}
            	Test target distribution: {'non-musk': 0.927681660899654, 'musk': 0.07231833910034602}
            Group Distribution molecule_name:
            	Train: 69
            	Test: 33
            
Train N: 5097, Test N: 1501
            Target Distribution:
            	Train target distribution: {'non-musk': 0.836962919364332, 'musk': 0.16303708063566805}
            	Test target distribution: {'non-musk': 0.8760826115922719, 'musk': 0.12391738840772819}
    

## Export

In [10]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019cb408-670c-7088-bf5e-eb09cb01e9b2
33fe3ecf263322e1a4636e9ebc56763a5c34392c4a6428f097a6f1cf693074e5
